# Notebook 6: Capstone - Build Your Own Decoder Model

In this final exercise, you will put everything together to build a functional, minimal text generation model from scratch, using the functional paradigms of JAX and the modular patterns seen in the `simply` codebase.

In [9]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', 'third-party', 'simply'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import jax
import jax.numpy as jnp
import numpy as np
import dataclasses
from simply.utils import module as module_lib
module_lib.ModuleRegistry.OVERWRITE_DUPLICATE = True
from simply.utils import common

print("Devices:", jax.devices())

Devices: [CpuDevice(id=0)]


## 1. Configuration
Let's define the hyper-parameters for our small decoder in a dataclass.

In [11]:
@dataclasses.dataclass
class ModelConfig:
    vocab_size: int = 1000
    dim: int = 128          # embedding/model dimension
    num_heads: int = 8
    head_dim: int = 16      # dim // num_heads
    ff_dim: int = 512       # 4 * dim
    max_seq_len: int = 64

config = ModelConfig()
print(config)

ModelConfig(vocab_size=1000, dim=128, num_heads=8, head_dim=16, ff_dim=512, max_seq_len=64)


## 2. Model Architecture
Let's define a simplified decoder block. In reality, models stack these blocks, but we will just write one Layer. You can use components like `module_lib.EinsumLinear` and `module_lib.EmbeddingLinear` from the Simply library.

In [32]:
@module_lib.ModuleRegistry.register
@dataclasses.dataclass
class SimpleDecoder(module_lib.SimplyModule):
    config: ModelConfig


    def setup(self):
        # TODO 1: Initialize the token embedding table.
        # Hint: module_lib.EmbeddingLinear takes vocab_size, dim
        # self.embed = ...
        self.embed = module_lib.EmbeddingLinear(
            vocab_size=self.config.vocab_size,
            dim=self.config.dim
        )

        # TODO 2: Initialize Linear projections for Q, K, V and Output (O) 
        # The einsum equations for Q, K, V should typically be '...d, dh -> ...h'
        # or if we map to heads: '...d, dhk -> ...hk'
        self.wq = module_lib.EinsumLinear(
            'dhk,...d->...hk',
            (self.config.dim, self.config.num_heads, self.config.head_dim),
        )
        
        self.wk = module_lib.EinsumLinear(
            'dhk,...d->...hk',
            (self.config.dim, self.config.num_heads, self.config.head_dim)
        )
        self.wv = module_lib.EinsumLinear(
            'dhk,...d->...hk',
            (self.config.dim, self.config.num_heads, self.config.head_dim),
        )

        self.wo = module_lib.EinsumLinear(
           'hkd,...hk->...d',
            (self.config.num_heads, self.config.head_dim, self.config.dim),
        )
        # TODO 3: Initialize FFN (two Linear layers with an activation in between)
        self.ffn1 = module_lib.EinsumLinear(
            "df,...d->...f",
            (self.config.dim, self.config.ff_dim),
        )

        self.ffn2 = module_lib.EinsumLinear(
            "fd,...f->...d",
            (self.config.ff_dim, self.config.dim),
        )

    def init(self, prng_key):
        keys = jax.random.split(prng_key, 7)
        params = {}
        params['embed'] = self.embed.init(keys[0])
        params['wq'] = self.wq.init(keys[1])
        params['wk'] = self.wk.init(keys[2])
        params['wv'] = self.wv.init(keys[3])
        params['wo'] = self.wo.init(keys[4])
        params['ffn1'] = self.ffn1.init(keys[5])
        params['ffn2'] = self.ffn2.init(keys[6])
        
        return params
        
    def apply(self, params, token_ids):
        # Input `token_ids` shape: (batch_size, seq_len)
        
        # TODO 4: Look up embeddings. (Call self.embed.embed)
        # x = ... 
        x = self.embed.embed(params['embed'], token_ids)
        
        # TODO 5: Calculate Q, K, V matrices and apply causal mask (Exercise 1 & 2 styles)
        # ...
        q = self.wq.apply(params['wq'], x)
        k = self.wk.apply(params['wk'], x)
        v = self.wv.apply(params['wv'], x)

        
        scale = jnp.sqrt(self.config.head_dim)
        attn_scores = jnp.einsum('bqhd,bkhd->bhqk', q, k) / scale
        seq_len = token_ids.shape[1]
        mask = jnp.tril(jnp.ones((seq_len, seq_len)))
        attn_scores = jnp.where(mask, attn_scores, -1e9)

        attn_weights = jax.nn.softmax(attn_scores, axis=-1)
        attn_out = jnp.einsum('bhqk,bkhd->bqhd', attn_weights, v)

        x = x + self.wo.apply(params['wo'], attn_out)
        # TODO 6: Pass output through FFN
        # ... 
        ffn_hidden = jax.nn.relu(self.ffn1.apply(params['ffn1'], x))
        x = x + self.ffn2.apply(params['ffn2'], ffn_hidden)
        
        # TODO 7: Map the output back to logits over the vocab.
        # Logits shape should be (batch_size, seq_len, vocab_size)
        # logits = ...
        logits = self.embed.apply(params['embed'], x)
        
        # return logits
        return logits

## 3. Initialization & Forward Pass
You need to instantiate your module and generate the initial PyTree of parameters using `init`.

In [33]:
# Pseudo-implementation (uncomment after finishing the above)

model = SimpleDecoder(config=config)
key = jax.random.PRNGKey(42)
params = model.init(key)

dummy_batch = jnp.zeros((8, config.max_seq_len), dtype=jnp.int32)
dummy_logits = model.apply(params, dummy_batch)
print("Params keys:", jax.tree_util.tree_map(lambda x: x.shape, params))
print("Logits shape:", dummy_logits.shape)

Params keys: {'embed': {'b': AnnotatedArray(array=(1000,), metadata=mappingproxy({'dim_annotation': '.'})), 'w': AnnotatedArray(array=(1000, 128), metadata=mappingproxy({'dim_annotation': '.i'}))}, 'ffn1': {'w': AnnotatedArray(array=(128, 512), metadata=mappingproxy({'dim_annotation': 'io'}))}, 'ffn2': {'w': AnnotatedArray(array=(512, 128), metadata=mappingproxy({'dim_annotation': 'io'}))}, 'wk': {'w': AnnotatedArray(array=(128, 8, 16), metadata=mappingproxy({'dim_annotation': 'ioo'}))}, 'wo': {'w': AnnotatedArray(array=(8, 16, 128), metadata=mappingproxy({'dim_annotation': 'iio'}))}, 'wq': {'w': AnnotatedArray(array=(128, 8, 16), metadata=mappingproxy({'dim_annotation': 'ioo'}))}, 'wv': {'w': AnnotatedArray(array=(128, 8, 16), metadata=mappingproxy({'dim_annotation': 'ioo'}))}}
Logits shape: (8, 64, 1000)


## 4. Loss & Data Generator
We provide a dummy data generator. You need to write the loss function that takes parameters and a batch, returning the scalar loss value. We use cross-entropy.

In [34]:
def get_dummy_batch(batch_size, key):
    return jax.random.randint(key, (batch_size, config.max_seq_len), 0, config.vocab_size)

# Remember: inputs are tokens without the last one, targets are tokens shifted by 1.
def compute_loss(params, tokens):
    # TODO 1: Extract inputs and targets from `tokens`
    # inputs = ...  # token[:, :-1]
    # targets = ... # token[:, 1:]
    inputs = tokens[:, :-1]
    targets = tokens[:, 1:]
    
    # TODO 2: Get logits from the model
    logits = model.apply(params, inputs)
    
    # TODO 3: Compute the softmax cross-entropy loss
    # Wait! jax handles this via Optax, but let's implement the math or use a utility.
    # (A simple way: one_hot target, multiply by log_softmax)
    # loss = ...
    one_hot_targets = jax.nn.one_hot(targets, config.vocab_size)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    token_losses = -jnp.sum(one_hot_targets * log_probs, axis=-1)
    
    # return loss.mean()
    return token_losses.mean()

## 5. Training Loop
Use `jax.value_and_grad` to create a compiled step function, then loop over random dummy batches to prove the logic fits together.

In [36]:
learning_rate = 1e-3

# TODO 1: Wrap `compute_loss` with point-in-time jax.value_and_grad to get `loss_and_grad_fn`
# loss_and_grad_fn = ...
loss_and_grad_fn = jax.value_and_grad(compute_loss)

@jax.jit
def train_step(params, batch):
    # TODO 2: compute `loss` and `grads`.
    # pass
    loss, grads = loss_and_grad_fn(params, batch)
    
    # TODO 3: Update params functionally (Exercise 5 style)
    # new_params = jax.tree_util.tree_map(lambda p, g: p - learning_rate * g, params, grads)
    new_params = jax.tree_util.tree_map(
        lambda p, g: p - learning_rate * g,
        params,
        grads
    )
    
    return params, loss

# -- The loop -- 
key = jax.random.PRNGKey(101)
for step in range(500):
    key, subkey = jax.random.split(key)
    batch = get_dummy_batch(16, subkey)
    params, loss_val = train_step(params, batch)
    if step % 10 == 0:
        print(f"Step {step}: Loss = {loss_val:.4f}")

Step 0: Loss = 11.2859
Step 10: Loss = 11.3524
Step 20: Loss = 11.2602
Step 30: Loss = 11.1894
Step 40: Loss = 11.2522
Step 50: Loss = 11.3058
Step 60: Loss = 11.2636
Step 70: Loss = 11.2688
Step 80: Loss = 11.3042
Step 90: Loss = 11.2309
Step 100: Loss = 11.2151
Step 110: Loss = 11.2724
Step 120: Loss = 11.2970
Step 130: Loss = 11.3836
Step 140: Loss = 11.3495
Step 150: Loss = 11.3795
Step 160: Loss = 11.2588
Step 170: Loss = 11.2865
Step 180: Loss = 11.3060
Step 190: Loss = 11.2127
Step 200: Loss = 11.1435
Step 210: Loss = 11.3039
Step 220: Loss = 11.2696
Step 230: Loss = 11.1610
Step 240: Loss = 11.2490
Step 250: Loss = 11.2812
Step 260: Loss = 11.2341
Step 270: Loss = 11.2498
Step 280: Loss = 11.2332
Step 290: Loss = 11.2322
Step 300: Loss = 11.3334
Step 310: Loss = 11.1882
Step 320: Loss = 11.2503
Step 330: Loss = 11.2519
Step 340: Loss = 11.3492
Step 350: Loss = 11.3538
Step 360: Loss = 11.2655
Step 370: Loss = 11.2248
Step 380: Loss = 11.1995
Step 390: Loss = 11.2679
Step 400: L